<a href="https://colab.research.google.com/github/abegithub2024/abegithub2024/blob/main/Soil_Loss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap
!pip install xee
import xee
import xarray as xr
import pandas as pd

In [ ]:
import ee
import geemap

# Trigger OAuth2 authentication flow
ee.Authenticate()
ee.Initialize(
    project = 'ee-abehegeno',
    opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
map = geemap.Map(basemap = 'SATELLITE')
map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [ ]:
# 1️⃣ Define Area of Interest (AOI)
# ===========================================================
aoi = ee.FeatureCollection('projects/ee-abehegeno/assets/Aari_woredas')

Map = geemap.Map(center=[6.2, 36.8], zoom=8)
Map.addLayer(aoi, {}, "AOI")


In [ ]:
# 2️⃣ Rainfall Erosivity Factor (R) using CHIRPS
# Equation: R = 0.562 * P - 8.12
# ===========================================================

# Load CHIRPS daily rainfall
chirps = (
    ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
    .filterDate('2020-01-01', '2020-12-31')
    .filterBounds(aoi)
)

# Compute total annual rainfall (mm)
annual_rainfall = chirps.select('precipitation').sum().clip(aoi)

# Compute R-factor using your formula
R = annual_rainfall.multiply(0.562).subtract(8.12).rename('R')

# Clip and remove negative R values (if P < 14.45 mm, R can become negative)
R = R.where(R.lt(0), 0).clip(aoi)

# Visualization
rVis = {'min': 0, 'max': 1000, 'palette': ['blue', 'green', 'yellow', 'red']}
Map.addLayer(R, rVis, 'R Factor (R=0.562×P−8.12)')


In [ ]:
def R_factor(image):
  """Placeholder function for R factor calculation."""
  pass

In [ ]:
# 3️⃣ Soil Erodibility Factor (K) using OpenLandMap
# ===========================================================
soil = ee.Image('OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02').clip(aoi)
soil_class = soil.select('b10').rename('soil_class')

K = soil_class.remap(
    ee.List.sequence(1, 12),
    ee.List([0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.05, 0.07, 0.09, 0.11, 0.13, 0.14])
).rename('K').clip(aoi)

Map.addLayer(K, {'min': 0, 'max': 0.15, 'palette': ['white', 'yellow', 'brown']}, 'K Factor')

In [ ]:
def K_factor(img):
  """Placeholder function for K factor calculation."""
  pass

In [ ]:
# ===========================================================
# 4️⃣ LS Factor (Fixed method using SRTM and HydroSHEDS)
# ===========================================================
dem = ee.Image('USGS/SRTMGL1_003').clip(aoi)
slope_deg = ee.Terrain.slope(dem)
slope_rad = slope_deg.multiply(3.1416 / 180)

# Hydrologically conditioned DEM (if available)
hydroDEM = ee.Image('WWF/HydroSHEDS/03VFDEM').clip(aoi)
flow_acc = hydroDEM.select('b1').rename('flowAccum')

# Fallback if HydroSHEDS not available
filled = dem.focal_min(1).focal_max(1)
fallback_acc = (
    ee.Terrain.hillshade(filled)
    .multiply(0)
    .add(1)
    .cumulativeCost(source=filled.gt(0), maxDistance=1000)
    .rename('flowAccum')
)
flow_acc = flow_acc.unmask(fallback_acc)

cell_size = ee.Image.pixelArea().sqrt()
flow_acc_m2 = flow_acc.multiply(ee.Image.pixelArea())
slope_length = flow_acc_m2.divide(cell_size)

m = slope_deg.expression(
    "(s < 1) ? 0.2"
    " : (s >= 1 && s < 3) ? 0.3"
    " : (s >= 3 && s < 5) ? 0.4"
    " : 0.5",
    {"s": slope_deg},
).rename('m')

LS = (
    slope_length.divide(22.13).pow(m)
    .multiply(slope_rad.sin().divide(0.0896).pow(1.3))
    .rename('LS')
)
LS_capped = LS.where(LS.gt(100), 100).clip(aoi)

Map.addLayer(LS_capped, {'min': 0, 'max': 10, 'palette': ['white', 'green', 'black']}, 'LS Factor')

In [ ]:
# 5️⃣ Cover Management Factor (C)
# ===========================================================
landcover = (
    ee.ImageCollection('MODIS/061/MCD12Q1')
    .filterDate('2020-01-01', '2020-12-31')
    .first()
    .select('LC_Type1')
    .clip(aoi)
)

C = landcover.remap(
    ee.List.sequence(0, 17),
    ee.List([0.001, 0.001, 0.001, 0.01, 0.01, 0.05, 0.05, 0.2, 0.3, 0.8, 0.001, 0.8, 0.9, 0.5, 0.4, 0.6, 0.02, 0.02])
).rename('C').clip(aoi)

Map.addLayer(C, {'min': 0, 'max': 1, 'palette': ['white', 'green', 'darkgreen']}, 'C Factor')


In [ ]:
# 6️⃣ Support Practice Factor (P)
# ===========================================================
P = ee.Image.constant(1).clip(aoi).rename('P')
Map.addLayer(P, {'min': 0, 'max': 1, 'palette': ['white']}, 'P Factor')

# ===========================================================
# 7️⃣ Compute RUSLE Soil Loss
# ===========================================================
A = R.multiply(K).multiply(LS_capped).multiply(C).multiply(P).rename('Soil_Loss')
Map.addLayer(A, {'min': 0, 'max': 200, 'palette': ['white', 'yellow', 'red']}, 'RUSLE Soil Loss')


In [ ]:
# ===========================================================
# 8️⃣ Summary Statistics
# ===========================================================
mean_soil_loss = A.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi.geometry(),
    scale=90,
    maxPixels=1e13
)
print("Mean Soil Loss (tons/ha/yr):", mean_soil_loss.getInfo())

# ===========================================================
# Display Map
# ===========================================================
Map.addLayerControl()
Map

Mean Soil Loss (tons/ha/yr): {'Soil_Loss': None}


Map(center=[6.2, 36.8], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(c…

In [ ]:
# 9️⃣ EXPORT RESULTS TO GOOGLE DRIVE
# ===========================================================
drive_folder = 'GEE_RUSLE_Exports'
scale = 90  # adjust for your DEM resolution

def export_to_drive(image, name):
    task = ee.batch.Export.image.toDrive(
        image=image.toFloat(),
        description=name,
        folder=drive_folder,
        fileNamePrefix=name,
        region=aoi.geometry(),
        scale=scale,
        crs='EPSG:4326',
        maxPixels=1e13
    )
    task.start()
    print(f'Export started: {name}')

export_to_drive(R, 'R_Factor_2020')
export_to_drive(K, 'K_Factor_2020')
export_to_drive(LS_capped, 'LS_Factor_2020')
export_to_drive(C, 'C_Factor_2020')
export_to_drive(P, 'P_Factor_2020')
export_to_drive(A, 'RUSLE_Soil_Loss_2020')

print('✅ All export tasks have been started. Check your Google Drive → GEE_RUSLE_Exports folder.')

Export started: R_Factor_2020
Export started: K_Factor_2020
Export started: LS_Factor_2020
Export started: C_Factor_2020
Export started: P_Factor_2020
Export started: RUSLE_Soil_Loss_2020
✅ All export tasks have been started. Check your Google Drive → GEE_RUSLE_Exports folder.
